In [ ]:
%%sql -r dataframe_1
USE DATABASE SKYPOINTS_INCU;

SHOW NOTEBOOKS IN DATABASE SKYPOINTS_INCU;

In [ ]:
USE DATABASE SKYPOINTS_INCU;

SHOW CODE BUNDLES IN DATABASE SKYPOINTS_INCU;

SHOW NOTEBOOK PROJECTS IN DATABASE SKYPOINTS_INCU;

In [ ]:
LIST 'snow://workspace/USER$.PUBLIC."incubyte_project"/versions/last/';

In [ ]:
SHOW GIT REPOSITORIES IN SCHEMA USER$.PUBLIC;

In [ ]:
SHOW COMPUTE POOLS;

In [ ]:
LIST 'snow://workspace/USER$.PUBLIC."incubyte_project"/versions/last/';

In [ ]:
USE DATABASE SKYPOINTS_INCU;
USE SCHEMA OPS;

CREATE CODE BUNDLE IF NOT EXISTS SKYPOINTS_PIPELINE_BUNDLE
FROM 'snow://workspace/USER$.PUBLIC."incubyte_project"/versions/version$9'
COMMENT = 'SkyPoints Incubyte end-to-end data engineering pipeline';

In [ ]:
DESCRIBE CODE BUNDLE SKYPOINTS_INCU.OPS.SKYPOINTS_PIPELINE_BUNDLE;

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

required_tables = [
    ("RAW", "RAW_MEMBER_PROFILES"),
    ("RAW", "RAW_REDEMPTIONS"),
    ("STG", "STG_MEMBER_PROFILES"),
    ("STG", "STG_REDEMPTION_TRANSACTIONS"),
    ("CORE", "MEMBER_PROFILE"),
    ("CORE", "REDEMPTION_TRANSACTIONS"),
    ("CORE", "MEMBER_PROFILE_IND"),
    ("CORE", "MEMBER_PROFILE_USA"),
    ("CORE", "MEMBER_PROFILE_PHIL"),
    ("CORE", "MEMBER_PROFILE_CAN"),
    ("CORE", "MEMBER_PROFILE_AU"),
    ("OPS", "INGESTION_AUDIT"),
    ("OPS", "BATCH_CONTROL"),
    ("OPS", "REJECTED_MEMBER_RECORDS"),
    ("OPS", "REJECTED_REDEMPTION_RECORDS"),
    ("OPS", "DATA_QUALITY_RESULTS"),
]

missing_tables = []

for schema_name, table_name in required_tables:
    result = session.sql(f"""
        SELECT COUNT(*) AS TABLE_COUNT
        FROM SKYPOINTS_INCU.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_SCHEMA = '{schema_name}'
          AND TABLE_NAME = '{table_name}'
    """).collect()

    if result[0]["TABLE_COUNT"] == 0:
        missing_tables.append(f"{schema_name}.{table_name}")

if missing_tables:
    raise RuntimeError(
        "Pipeline preflight failed. Missing tables: "
        + ", ".join(missing_tables)
    )

print("PIPELINE PREFLIGHT PASSED")
print(f"Database: SKYPOINTS_INCU")
print(f"Required tables verified: {len(required_tables)}")